# CAS Exam 5: Expected Claims Method (Using Friedland Industry Auto Data)

**Source:** Friedland, J. *Estimating Unpaid Claims Using Basic Techniques*, Casualty Actuarial Society, 2010

This notebook uses:
- `C:/Users/cphel/Documents/code/exam_5/chainladder-python/chainladder/utils/data/friedland_us_industry_auto.csv`

Learning goals:
- Show expected claims method concepts with the same dataset used in the chain ladder notebook.
- Use `chainladder.Triangle` for the claims side, then apply ECR/pure-premium priors for ultimate and IBNR.
- Compare provided ECR vs derived ECR approaches with actuarial considerations.

## Formula-Sheet Core Equations

Premium basis:
- Ultimate Claims = (Expected Claims / Earned Premium) x Earned Premium
- ECR = Expected Claims / Earned Premium

Exposure basis:
- Ultimate Claims = (Expected Claims / Earned Exposure) x Earned Exposure
- Expected Pure Premium = Expected Claims / Earned Exposure

Interpretation:
- Expected claims method uses a priori assumptions (ECR/pure premium) more heavily than immature emergence patterns.


---
## Formula-Sheet Reference: Expected Claims Method

### Process Overview (Friedland Ch. 5)

| Step | Action | Key Actuarial Decision |
|------|--------|------------------------|
| 1 | Obtain earned premium (or exposure) by AY | Must be on a consistent rate/coverage basis |
| 2 | Select ECR (or Pure Premium) | Provided a priori, or derived from historical experience |
| 3 | Apply level adjustments if needed | Rate changes, operational changes, coverage shifts |
| 4 | Compute expected ultimate: `Ultimate = ECR × EP` | Or `PP × Exposure` on exposure basis |
| 5 | Calculate IBNR: `IBNR = Ultimate − Paid (or Reported)` | Paid basis gives larger IBNR; reported basis is more common |

### Key Formulas

**Premium basis:**
$$\text{Expected Ultimate}_i = \text{ECR} \times \text{Earned Premium}_i$$
$$\text{ECR} = \frac{\text{Expected Claims}}{\text{Earned Premium}}$$

**Exposure basis:**
$$\text{Expected Ultimate}_i = \overline{PP} \times \text{Earned Exposure}_i$$
$$\overline{PP} = \frac{\text{Expected Claims}}{\text{Earned Exposure}}$$

**IBNR (paid basis):**
$$\text{IBNR}_i = \text{Expected Ultimate}_i - \text{Paid to Date}_i$$

**IBNR (reported basis):**
$$\text{IBNR}_i = \text{Expected Ultimate}_i - \text{Reported to Date}_i$$

### ECR Derivation Approaches

| Method | Formula | When to Use |
|--------|---------|-------------|
| Arithmetic average | $\bar{r} = \frac{1}{n}\sum r_i$ | Equal weight per AY; best when premium volumes are similar across years |
| Median | $\text{med}(r_1, \ldots, r_n)$ | Robust to a single outlier year |
| Volume-weighted | $\frac{\sum \text{Claims}_i}{\sum \text{Premium}_i}$ | **Exam default — preferred when premium volumes differ significantly across AYs** |

### Level Adjustment

If the historical ECR was calibrated to an older rate or coverage level:
$$\text{Adjusted ECR} = \text{Historical ECR} \times \text{Level Adjustment Factor}$$

### Connection to BF and Chain Ladder

The three methods form a credibility spectrum:

$$\underbrace{\text{Expected Claims}}_{\text{100\% prior weight}} \longrightarrow \underbrace{\text{BF}}_{\text{blend}} \longrightarrow \underbrace{\text{Chain Ladder}}_{\text{100\% emergence weight}}$$

**BF formula:**
$$\text{BF Ultimate}_i = \text{Reported}_i + \underbrace{(1 - \hat{q}_i)}_{\text{\% unreported}} \times \underbrace{\text{Expected Ultimate}_i}_{\text{a priori}}$$

- When $\hat{q}_i \approx 0$ (very immature): BF $\approx$ Expected Ultimate — prior dominates
- When $\hat{q}_i \approx 1$ (very mature): BF $\approx$ Reported — chain ladder dominates

> **Exam insight:** The expected claims method is the limiting case of BF where 100% weight is placed on the a priori. BF is almost always preferred over pure expected claims unless the line is brand new (no development history whatsoever).

### When to Use Expected Claims (vs Chain Ladder vs BF)

| Situation | Preferred Method | Reason |
|-----------|-----------------|--------|
| Very immature AY (≤ 24 months) | **Expected Claims** | CDF enormous; leverage risk dominates actual data |
| New line or new territory | **Expected Claims** | No credible development history exists |
| Major operational change | **Expected Claims** | Historical LDFs no longer applicable |
| Moderate maturity (24–60 months) | **BF** | Growing credibility of actual emergence; blend is optimal |
| Mature, stable AY | **Chain Ladder** | Actual development is fully credible |

### Key Assumptions
1. Selected ECR reliably represents the long-run loss ratio for each projection AY
2. Earned premium is on a consistent rate and coverage basis
3. Level adjustments correctly account for rate changes between calibration and projection years
4. The method assigns **equal weight** to each AY regardless of maturity — no credibility weighting by development age

In [ ]:
from __future__ import annotations

from pathlib import Path

import chainladder as cl
import pandas as pd

ROOT = Path.cwd().resolve()

from reservingengine.reserving import (
    adjust_selected_ecr,
    build_environment_impact_table,
    expected_claims_from_exposure,
    expected_claims_from_premium,
    ibnr_from_expected_claims,
    implied_ecr,
    selected_ecr_from_history,
)

DATA_PATH = ROOT / 'chainladder-python' / 'chainladder' / 'utils' / 'data' / 'friedland_us_industry_auto.csv'
raw = pd.read_csv(DATA_PATH)

triangle = cl.Triangle(
    raw,
    origin='Accident Year',
    development='Calendar Year',
    columns=['Paid Claims', 'Reported Claims'],
    cumulative=True,
)
paid_triangle = triangle['Paid Claims']
reported_triangle = triangle['Reported Claims']

raw.head(), {'triangle_shape': triangle.shape, 'valuation_date': str(triangle.valuation_date)}


## Step 1: Load Data and Understand Development Status

The Friedland industry auto dataset provides paid and reported claims for accident years 1998–2007 across 10 development ages (12–120 months). Unlike the chain ladder method, expected claims **does not project development factors** — it replaces that process entirely with a selected ECR.

However, understanding how developed each AY is serves two purposes:
1. **Identify credible calibration years** — only mature AYs (near their ultimate) should anchor the historical ECR derivation
2. **Understand IBNR composition** — immature AYs will have the largest IBNR as a share of expected ultimate, and are where the expected claims method is most important

The table below shows each AY's latest diagonal position at valuation date 12/31/2007.

In [ ]:
paid_long = paid_triangle.to_frame(origin_as_datetime=False, keepdims=True).reset_index()
reported_long = reported_triangle.to_frame(origin_as_datetime=False, keepdims=True).reset_index()
paid_matrix = paid_long.pivot(index='origin', columns='development', values='Paid Claims').sort_index().sort_index(axis=1)
reported_matrix = reported_long.pivot(index='origin', columns='development', values='Reported Claims').sort_index().sort_index(axis=1)

latest_paid = paid_triangle.latest_diagonal.to_frame().iloc[:, 0]
latest_reported = reported_triangle.latest_diagonal.to_frame().iloc[:, 0]
latest_age = paid_matrix.notna().iloc[:, ::-1].idxmax(axis=1)

claim_inputs = pd.DataFrame(
    {
        'LatestPaid': latest_paid.values,
        'LatestReported': latest_reported.values,
        'LatestDevelopmentAge': latest_age.values,
    },
    index=latest_paid.index.year,
)
claim_inputs.index.name = 'AccidentYear'
claim_inputs


### Development Status: % Paid of Ultimate

For context, a quick volume-weighted chain ladder is run to estimate ultimate and compute `% Paid` and `% Unreported` for each AY. This is **not** the expected claims method — it is purely diagnostic, illustrating why the expected claims approach is appropriate for the most recent years.

> **Exam connection:** High `% Unreported` = high chain ladder leverage = expected claims (or BF) preferred over pure development. AY 2007 at 12 months has the largest unreported fraction; applying a chain ladder CDF there would amplify any LDF selection error dramatically.

In [ ]:
# Quick chain ladder ultimate estimate — used only to compute % paid / % unreported for context
# This shows WHY expected claims is needed for immature AYs: very high unreported fraction
_dev_fitted = cl.Development(average='volume').fit_transform(paid_triangle)
_cl_ult_frame = cl.Chainladder().fit(_dev_fitted).ultimate_.to_frame()
_cl_ult = _cl_ult_frame.iloc[:, 0]
_cl_ult.index = [int(str(x)[:4]) for x in _cl_ult.index]  # normalize to integer years

development_status = pd.DataFrame({
    'DevAge': claim_inputs['LatestDevelopmentAge'],
    'Paid': claim_inputs['LatestPaid'],
    'CL_Ultimate_Est': _cl_ult,
    'PctPaid': claim_inputs['LatestPaid'] / _cl_ult,
    'PctUnreported': 1 - claim_inputs['LatestPaid'] / _cl_ult,
    'PaidToReported': claim_inputs['LatestPaid'] / claim_inputs['LatestReported'],
})

development_status.style.format({
    'Paid': '{:,.0f}',
    'CL_Ultimate_Est': '{:,.0f}',
    'PctPaid': '{:.1%}',
    'PctUnreported': '{:.1%}',
    'PaidToReported': '{:.1%}',
}).background_gradient(subset=['PctUnreported'], cmap='RdYlGn_r')

## Build Priors (Earned Premium / Exposure) Aligned to This Dataset

This CSV has paid and reported claims but not premium/exposure. For expected-claims demonstration, we add explicit prior assumptions by AY:
- pricing ECR assumption to infer earned premium scale,
- average premium per exposure to infer earned exposure.

These priors are the central modeling choice in the expected claims method.


In [ ]:
ay = claim_inputs.index

pricing_ecr_assumption = pd.Series(
    [0.74, 0.74, 0.745, 0.75, 0.755, 0.76, 0.765, 0.77, 0.775, 0.78],
    index=ay,
    dtype=float,
)
earned_premium = claim_inputs['LatestReported'] / pricing_ecr_assumption

avg_premium_per_exposure = pd.Series(
    [4700, 4750, 4800, 4850, 4900, 4950, 5000, 5050, 5100, 5150],
    index=ay,
    dtype=float,
)
earned_exposure = earned_premium / avg_premium_per_exposure

prior_inputs = pd.DataFrame(
    {
        'PricingECRAssumption': pricing_ecr_assumption,
        'EarnedPremium': earned_premium,
        'AvgPremiumPerExposure': avg_premium_per_exposure,
        'EarnedExposure': earned_exposure,
    }
)
prior_inputs


## Provided ECR Case and Exposure-Basis Cross-Check

Exam guideline reminder:
- If provided ECR is not tied to a specific AY, treat it as broadly applicable unless stated otherwise.

We project expected ultimate with a provided ECR, then show the equivalent exposure-basis framing.


In [ ]:
provided_ecr = 0.78
expected_ultimate_provided = expected_claims_from_premium(earned_premium, provided_ecr)

selected_pure_premium = float(expected_ultimate_provided.sum() / earned_exposure.sum())
expected_ultimate_exposure = expected_claims_from_exposure(earned_exposure, selected_pure_premium)

provided_view = pd.DataFrame(
    {
        'EarnedPremium': earned_premium,
        'ExpectedUltimate_ProvidedECR': expected_ultimate_provided,
        'ExpectedUltimate_ExposureBasis': expected_ultimate_exposure,
    }
)
provided_view.loc['Total'] = provided_view.sum()
provided_view


### Provided ECR: Actuarial Considerations

**Premium vs exposure basis:** Both approaches produce the same **aggregate** expected ultimate — they are mathematically equivalent at the total level. Differences appear at the AY level when the average premium per exposure shifts over time (as it does here, trending from \$4,700 to \$5,150).

**When is a flat provided ECR appropriate?**
- Regulatory rate filings that specify a target loss ratio
- Management-directed a priori assumption for a new or restructured line
- Reinsurance pricing where the cedant provides a blended industry loss ratio

**Sensitivity risk of a flat ECR:** Applying one ECR to all AYs ignores any trend in loss ratios over time. If the true ECR is drifting (social inflation, medical cost trends, mix changes), a flat ECR will systematically over- or under-state ultimates for early vs late AYs. This is why year-specific ECRs (as shown in the final comparison section) are often more defensible.

## Determine ECR from Historical Experience

Formula-sheet process:
1. Adjust historical data as needed.
2. Calculate claim ratios.
3. Select ECR (arithmetic, median, volume-weighted).

Here we use mature AYs (development age >= 84) and latest reported as a proxy for near-ultimate historical claims.


In [ ]:
mature_mask = claim_inputs['LatestDevelopmentAge'] >= 84
historical_claims_proxy = claim_inputs.loc[mature_mask, 'LatestReported']
historical_premium = earned_premium.loc[mature_mask]

historical_ecr = implied_ecr(historical_claims_proxy, historical_premium)
selected_ecrs = pd.Series(
    {
        'ArithmeticECR': selected_ecr_from_history(historical_claims_proxy, historical_premium, method='arithmetic'),
        'MedianECR': selected_ecr_from_history(historical_claims_proxy, historical_premium, method='median'),
        'VolumeWeightedECR': selected_ecr_from_history(historical_claims_proxy, historical_premium, method='volume_weighted'),
    }
)

historical_review = pd.DataFrame(
    {
        'HistoricalClaimsProxy': historical_claims_proxy,
        'HistoricalPremium': historical_premium,
        'ImpliedHistoricalECR': historical_ecr,
    }
)
historical_review, selected_ecrs


### Step 3: ECR Selection — Actuarial Judgment

The three selection methods above typically give similar results for stable lines. Key exam judgment points:

**Why volume-weighted is the exam default:**
- Gives implicit credibility proportional to earned premium size
- Higher-premium years (often more recent) receive more weight — appropriate if the line is growing or rates have changed
- Most robust to years with abnormal volume

**Calibration window judgment:**
- Only use **mature AYs** (development age ≥ 84 months for auto liability) — immature AYs have unreported claims that bias the implied ECR downward
- In this dataset, AYs 1998–2001 (84–120 months at valuation) are credible calibration years; AYs 2002+ are excluded because they are still developing
- A 4–5 year window is typical; longer windows introduce trend distortion from older rate levels

**When derived ECR differs significantly from provided ECR:**
- Investigate whether rate level changes between calibration years and projection years explain the gap
- Apply an on-level adjustment before comparing (this is the "level adjustment" step below)
- If both are reasonable, the difference quantifies ECR parameter uncertainty — a key driver of reserve range

## Apply ECR Level Adjustment and Project IBNR

General guideline from the sheet:
- If premium level differs between historical calibration and projection AY, adjust ECR level before applying.

Below uses volume-weighted historical ECR with a 3% level adjustment.


In [ ]:
base_ecr = float(selected_ecrs['VolumeWeightedECR'])
level_adjustment = 1.03
adjusted_ecr = adjust_selected_ecr(base_ecr, level_adjustment=level_adjustment)

expected_ultimate_derived = expected_claims_from_premium(earned_premium, adjusted_ecr)
paid_to_date = claim_inputs['LatestPaid']
ibnr_derived = ibnr_from_expected_claims(expected_ultimate_derived, paid_to_date)

derived_projection = pd.DataFrame(
    {
        'PaidToDate': paid_to_date,
        'ExpectedUltimate_Derived': expected_ultimate_derived,
        'IBNR_Derived': ibnr_derived,
    }
)
derived_projection.loc['Total'] = derived_projection.sum()
pd.Series({'BaseECR': base_ecr, 'LevelAdjustment': level_adjustment, 'AdjustedECR': adjusted_ecr}), derived_projection


### Step 5: Extended Projection Table — IBNR Composition by AY

This table mirrors the chain ladder "extended projection" format. The `% Paid` and `% Unreported` columns show how much of the expected ultimate is still outstanding for each AY.

**Key observations:**
- `% Unreported` is high for immature AYs — most of the reserve is driven by the a priori ECR, not by actual development
- For mature AYs (small % unreported), the IBNR is small and corresponds mostly to case-outstanding run-off
- The `% Unreported` column equals the $(1 - \hat{q}_i)$ weight in the BF formula — this is how BF "blends" the two methods

**Paid vs reported basis:**  
Using paid-to-date gives larger IBNR because paid lags reported. Using reported-to-date is more common in practice (and in Friedland exam problems) because it credits case reserves already established.

In [ ]:
# Extended projection table: mirrors the chain ladder "extended" format
# Adds % Paid and % Unreported columns relative to Expected Ultimate
# These correspond directly to the (1 - q) term in the BF formula

exp_ult = expected_ultimate_derived.copy()
ibnr_reported_basis = ibnr_from_expected_claims(exp_ult, claim_inputs['LatestReported'])

extended = pd.DataFrame({
    'EarnedPremium': earned_premium,
    'ExpectedUltimate': exp_ult,
    'PaidToDate': claim_inputs['LatestPaid'],
    'ReportedToDate': claim_inputs['LatestReported'],
    'IBNR_PaidBasis': ibnr_derived,
    'IBNR_ReportedBasis': ibnr_reported_basis,
    'PctPaid': claim_inputs['LatestPaid'] / exp_ult,
    'PctUnreported': 1 - claim_inputs['LatestPaid'] / exp_ult,
})

totals = extended[['EarnedPremium', 'ExpectedUltimate', 'PaidToDate',
                    'ReportedToDate', 'IBNR_PaidBasis', 'IBNR_ReportedBasis']].sum()
extended.loc['Total'] = totals

(
    pd.Series({
        'Applied ECR (adjusted)': adjusted_ecr,
        'Total IBNR — Paid Basis ($M)': ibnr_derived.sum() / 1e6,
        'Total IBNR — Reported Basis ($M)': ibnr_reported_basis.sum() / 1e6,
    }).round(4),
    extended.style.format({
        'EarnedPremium': '{:,.0f}',
        'ExpectedUltimate': '{:,.0f}',
        'PaidToDate': '{:,.0f}',
        'ReportedToDate': '{:,.0f}',
        'IBNR_PaidBasis': '{:,.0f}',
        'IBNR_ReportedBasis': '{:,.0f}',
        'PctPaid': '{:.1%}',
        'PctUnreported': '{:.1%}',
    }).background_gradient(subset=['PctUnreported'], cmap='RdYlGn_r')
)

## Assumptions, Use Cases, and Environmental Impacts

Key assumptions:
1. A priori estimate is more reliable than immature emergence.
2. Current paid/reported to date may have limited predictive power for ultimate on young AYs.

Works well when:
- new line/territory has limited historical development credibility,
- operational changes reduce comparability of historical development,
- early maturity development factors are highly leveraged.


In [ ]:
impact_table = build_environment_impact_table()
impact_table


---
## Bornhuetter-Ferguson Connection: Expected Claims as the A Priori

The expected claims method is the foundation of the BF method. Understanding this connection is one of the most important concepts for Exam 5.

### BF Formula

$$\text{BF Ultimate}_i = \text{Reported}_i + \underbrace{(1 - \hat{q}_i)}_{\text{\% unreported}} \times \underbrace{\text{Expected Ultimate}_i}_{\text{a priori from expected claims}}$$

where $\hat{q}_i$ = cumulative % paid (or % reported) at latest evaluation for AY $i$.

### Credibility Spectrum

| Method | Formula | Weight on A Priori | Weight on Actual Emergence |
|--------|---------|-------------------|----------------------------|
| Expected Claims | $\text{Exp. Ultimate}$ | **100%** | 0% |
| BF | $\text{Reported} + (1-q) \times \text{Exp. Ult}$ | $(1-q) \times 100\%$ | $q \times 100\%$ |
| Chain Ladder | $\text{Reported} \times \text{CDF}$ | 0% | **100%** |

> As an AY matures ($q \to 1$), BF naturally converges to chain ladder. For brand-new AYs ($q \approx 0$), BF produces almost the same answer as expected claims.

### Worked Example (AY 2007 at 12 months, % paid ≈ 57% of expected ultimate)

Using `adjusted_ecr ≈ 0.7663` and `earned_premium(2007) ≈ $62.6M`:

$$\text{Expected Ultimate}_{2007} = 0.7663 \times \$62.6M \approx \$48.0M$$
$$\text{BF Ultimate}_{2007} = \$48.9M_{\text{reported}} + (1 - 0.57) \times \$48.0M \approx \$69.5M$$

Compare:
- **Expected Claims alone:** $\$48.0M$ — 100% weight on prior, ignores that $\$48.9M$ is already reported
- **BF:** $\$69.5M$ — credits the $\$48.9M$ reported plus projects remaining unreported via the prior
- **Chain Ladder:** would apply a very large CDF to just $\$27.2M$ paid — highly leveraged

> **Exam insight:** For AY 2007, BF is clearly better than either extreme. Expected claims is used here only to supply the a priori input. Recognizing when to use which method — and articulating why — is a common exam question.

## Provided vs Derived ECR Comparison

Educational takeaway:
- Provided ECR approach is straightforward and transparent but sensitive to whether the supplied ratio reflects current conditions.
- Derived ECR approach is anchored in historical calibration but sensitive to calibration window, mix changes, and level adjustments.


In [ ]:
year_specific_ecr = pd.Series(
    [0.77, 0.77, 0.775, 0.78, 0.785, 0.79, 0.795, 0.80, 0.805, 0.81],
    index=ay,
    dtype=float,
)

expected_ultimate_year_specific = expected_claims_from_premium(earned_premium, year_specific_ecr)
ibnr_provided = ibnr_from_expected_claims(expected_ultimate_provided, paid_to_date)
ibnr_year_specific = ibnr_from_expected_claims(expected_ultimate_year_specific, paid_to_date)

comparison = pd.DataFrame(
    {
        'Ultimate_ProvidedECR': expected_ultimate_provided,
        'Ultimate_DerivedAdjustedECR': expected_ultimate_derived,
        'Ultimate_YearSpecificECR': expected_ultimate_year_specific,
        'IBNR_ProvidedECR': ibnr_provided,
        'IBNR_DerivedAdjustedECR': ibnr_derived,
        'IBNR_YearSpecificECR': ibnr_year_specific,
    }
)
comparison.loc['Total'] = comparison.sum()
comparison


---
## ECR Sensitivity Analysis: Reserve Range Across ECR Assumptions

The expected claims method's primary sensitivity is the selected ECR. Unlike chain ladder — where sensitivity flows through LDF selection and propagates differently by AY maturity — expected claims has a **direct, linear, and uniform** relationship between ECR and ultimate:

$$\Delta \text{Ultimate}_i = \Delta \text{ECR} \times \text{Earned Premium}_i$$

A 1% increase in ECR increases every AY's ultimate by exactly 1%, regardless of development age. This makes the reserve range easy to quantify but also means there is no "self-correcting" signal from actual claims emergence.

The sweep below shows how total IBNR and the IBNR for the three most immature AYs (2005–2007) vary across a plausible ECR range. The two reference points (derived + adjusted ECR and the flat provided ECR) should fall within this range.

In [ ]:
import numpy as np

# Sweep ECR from 0.70 to 0.86 in steps of 0.01
# Shows total IBNR (paid basis) and IBNR for the three most immature AYs (2005-2007)
ecr_sweep = np.round(np.arange(0.70, 0.87, 0.01), 4)
sensitivity_rows = []
for ecr_val in ecr_sweep:
    ult = expected_claims_from_premium(earned_premium, ecr_val)
    ibnr_series = ibnr_from_expected_claims(ult, claim_inputs['LatestPaid'])
    sensitivity_rows.append({
        'ECR': ecr_val,
        'TotalIBNR_M': ibnr_series.sum() / 1e6,
        'IBNR_AY2005_2007_M': ibnr_series.iloc[-3:].sum() / 1e6,
    })

sensitivity_df = pd.DataFrame(sensitivity_rows).set_index('ECR')

# Mark the two reference ECR values
ref_ecrs = {f'Derived+Adj ({adjusted_ecr:.4f})': adjusted_ecr, f'Provided ({provided_ecr:.2f})': provided_ecr}
print("Reference ECR assumptions:")
for label, val in ref_ecrs.items():
    closest = sensitivity_df.index[abs(sensitivity_df.index - val).argmin()]
    row = sensitivity_df.loc[closest]
    print(f"  {label:35s}  Total IBNR: ${row['TotalIBNR_M']:.1f}M  |  AY 2005-2007 IBNR: ${row['IBNR_AY2005_2007_M']:.1f}M")

print(f"\nReserve range across ECR {ecr_sweep.min():.2f}–{ecr_sweep.max():.2f}:")
print(f"  Total IBNR:         ${sensitivity_df['TotalIBNR_M'].min():.1f}M – ${sensitivity_df['TotalIBNR_M'].max():.1f}M")
print(f"  AY 2005-2007 IBNR:  ${sensitivity_df['IBNR_AY2005_2007_M'].min():.1f}M – ${sensitivity_df['IBNR_AY2005_2007_M'].max():.1f}M")
print()

sensitivity_df.rename(columns={
    'TotalIBNR_M': 'Total IBNR ($M)',
    'IBNR_AY2005_2007_M': 'IBNR AY2005–2007 ($M)',
}).style.format('{:.1f}').background_gradient(cmap='RdYlGn_r')

---
## Expected Claims Method: Exam Summary

### Strengths
- **Stable for immature AYs** — avoids the leverage problem inherent in chain ladder for young accident years
- **Transparent** — IBNR is directly proportional to ECR; easy to audit and explain to management
- **Appropriate for new lines** — when no development history exists, it is often the only credible option
- **Immune to development volatility** — case reserve changes, settlement rate shifts, and diagonal effects do not affect the ultimate estimate

### Weaknesses
- **100% weight on prior** — ignores actual claims emergence entirely, even when that data is informative
- **ECR sensitivity** — a 1% change in ECR produces a 1% change in every AY's ultimate with no differentiation by maturity
- **Level adjustment complexity** — requires careful on-leveling if historical ECR was calibrated to different rate levels
- **No self-correcting mechanism** — if the a priori was wrong, the method gives no signal; the error persists until the AY matures

### Exam Red Flags (when NOT to use expected claims)
- AY is well-developed (≥ 84 months for typical casualty lines) — chain ladder would use actual data more credibly
- You have reliable, stable development history with no operational changes
- The selected ECR is known to be stale or is inconsistent with current pricing levels

### Method Hierarchy for Exam 5

```
Less Mature ← AY Development Age → More Mature
Expected Claims  →  BF  →  Chain Ladder
(100% prior)      (blend)  (100% emergence)
```

| AY Maturity | Preferred Method | Rationale |
|-------------|-----------------|-----------|
| ≤ 24 months | Expected Claims | Development leverage too high to rely on actual emergence |
| 24–60 months | BF | Credibility blend of prior and actual emergence |
| ≥ 60 months | Chain Ladder | Actual development data is credible and dominant |
| Any — new line | Expected Claims | No development history available |

---
# Exam 5 Practice Problems — Expected Claims Method

The following problems mirror Exam 5 written question format.
Work through each calculation before checking the solution code.
All arithmetic is reproducible without code.

**Instructions:** Show all work. Round ELRs to 3 decimal places. Round dollar amounts to nearest whole number.


## Practice Problem 1: Pure Expected Claims Calculation

**Exam-style question (Format 1 — mechanical, 2–3 points):**

An actuary is estimating reserves for an auto liability book. The selected ELR is **0.750** for all accident years. The following data is available:

| AY   | Earned Premium (000s) | Reported Losses to Date (000s) |
|------|----------------------:|--------------------------------:|
| 2021 | 20,000                | 18,500                          |
| 2022 | 22,000                | 17,600                          |
| 2023 | 24,000                | 14,400                          |
| 2024 | 26,000                | 9,100                           |
| 2025 | 28,000                | 4,200                           |

**(a)** Calculate the Expected Claims ultimate for each accident year.  
**(b)** Calculate IBNR (reported basis) for each accident year.  
**(c)** Which accident year has the highest IBNR as a percentage of its expected ultimate? Explain why in one sentence.  


In [ ]:
import pandas as pd

# PRACTICE PROBLEM 1 SOLUTION
elr = 0.750

data = {
    'AY':       [2021,   2022,   2023,   2024,   2025],
    'Premium':  [20000,  22000,  24000,  26000,  28000],
    'Reported': [18500,  17600,  14400,   9100,   4200],
}
df = pd.DataFrame(data).set_index('AY')

# PART (a): Expected Ultimate = Premium x ELR
df['Exp_Ultimate'] = df['Premium'] * elr

# PART (b): IBNR = Expected Ultimate - Reported  (reported basis)
df['IBNR'] = df['Exp_Ultimate'] - df['Reported']

# PART (c): % Unreported = IBNR / Expected Ultimate
df['Pct_Unreported'] = df['IBNR'] / df['Exp_Ultimate']

# Add totals row
totals = {'Premium': df['Premium'].sum(),
          'Reported': df['Reported'].sum(),
          'Exp_Ultimate': df['Exp_Ultimate'].sum(),
          'IBNR': df['IBNR'].sum(),
          'Pct_Unreported': df['IBNR'].sum() / df['Exp_Ultimate'].sum()}
result = df.copy()
result.loc['Total'] = totals

print('=== PRACTICE PROBLEM 1 SOLUTION ===')
print(f'  Selected ELR: {elr:.3f}')
print()
print(result.to_string(float_format=lambda x: f'{x:,.1f}'))
print()

max_ay = df['Pct_Unreported'].idxmax()
print(f'PART (c): AY {max_ay} has the highest % unreported = '
      f'{df.loc[max_ay, "Pct_Unreported"]*100:.1f}%')
print()
print(
    'MODEL ANSWER:\n'
    'AY 2025 has the highest IBNR as a % of expected ultimate because it is '
    'at the earliest development age -- most claims have not yet been reported, '
    'so reported losses represent only a small fraction of the expected ultimate.\n'
    '\n'
    'KEY CHECKS:\n'
    '  IBNR = Expected Ultimate - Reported  (not just the Expected Ultimate)\n'
    '  Reported basis is most common on exam; paid basis gives higher IBNR\n'
    '  % Unreported = IBNR / Expected Ultimate = the BF prior weight'
)


## Practice Problem 2: Comparison Question — CL vs. Expected Claims

**Exam-style question (Format 2 — judgment, 6–7 points):**

An actuary is estimating reserves for a commercial liability book with five accident years. The selected ELR is **0.680**. The following data is available:

| AY   | Earned Premium (000s) | Reported to Date (000s) | CDF to Ultimate |
|------|----------------------:|------------------------:|----------------:|
| 2021 | 15,000                | 13,200                  | 1.050           |
| 2022 | 16,500                | 12,870                  | 1.120           |
| 2023 | 18,000                | 11,880                  | 1.250           |
| 2024 | 19,500                | 9,750                   | 1.600           |
| 2025 | 21,000                | 5,040                   | 3.200           |

**(a)** Calculate the Chain Ladder ultimate for **all** accident years. *(3 pts)*  
**(b)** Calculate the Expected Claims ultimate for **all** accident years. *(2 pts)*  
**(c)** For AY 2025, the two methods produce materially different results. Compare the two methods and recommend which to use for AY 2025. Justify your answer in 3–4 sentences. *(2 pts)*  


In [ ]:
import pandas as pd

# PRACTICE PROBLEM 2 SOLUTION
elr = 0.680

data = {
    'AY':       [2021,   2022,   2023,   2024,   2025],
    'Premium':  [15000,  16500,  18000,  19500,  21000],
    'Reported': [13200,  12870,  11880,   9750,   5040],
    'CDF':      [1.050,  1.120,  1.250,  1.600,  3.200],
}
df = pd.DataFrame(data).set_index('AY')

# PART (a): Chain Ladder ultimate = Reported x CDF
df['CL_Ultimate'] = df['Reported'] * df['CDF']

# PART (b): Expected Claims ultimate = Premium x ELR
df['EC_Ultimate'] = df['Premium'] * elr

# IBNR for each method (reported basis)
df['CL_IBNR'] = df['CL_Ultimate'] - df['Reported']
df['EC_IBNR'] = df['EC_Ultimate'] - df['Reported']

# % paid (from CDF)
df['Pct_Paid'] = 1 / df['CDF']

totals_cl = df['CL_Ultimate'].sum()
totals_ec = df['EC_Ultimate'].sum()

print('=== PART (a) & (b): CHAIN LADDER vs EXPECTED CLAIMS ===')
print(f'  Selected ELR: {elr:.3f}')
print()
display_cols = ['Reported', 'CDF', 'Pct_Paid', 'CL_Ultimate', 'EC_Ultimate', 'CL_IBNR', 'EC_IBNR']
print(df[display_cols].to_string(float_format=lambda x: f'{x:,.1f}'))
print(f'\n  Total CL Ultimate: {totals_cl:,.0f}')
print(f'  Total EC Ultimate: {totals_ec:,.0f}')
print()

ay = 2025
cl_ult = df.loc[ay, 'CL_Ultimate']
ec_ult = df.loc[ay, 'EC_Ultimate']
pct_paid = df.loc[ay, 'Pct_Paid']
print(f'=== AY {ay} COMPARISON ===')
print(f'  % Paid:              {pct_paid*100:.1f}%  (CDF = {df.loc[ay, "CDF"]:.3f})')
print(f'  CL Ultimate:         {cl_ult:>10,.0f}  (Reported x CDF)')
print(f'  EC Ultimate:         {ec_ult:>10,.0f}  (Premium x ELR)')
print(f'  Difference:          {cl_ult - ec_ult:>+10,.0f}')
print()
print('PART (c) -- MODEL ANSWER (exam format):')
print('-' * 60)
print(
    'AY 2025 is at only 12 months of development with just 31.3% of\n'
    'expected claims paid. The Chain Ladder applies a CDF of 3.200,\n'
    'meaning the entire ultimate is 3.2x the reported amount -- any\n'
    'error in LDF selection is amplified dramatically.\n'
    '\n'
    'The Expected Claims method avoids this leverage problem entirely by\n'
    'ignoring immature data and relying on the a priori ELR. For a brand-new\n'
    'accident year, the ELR is more credible than the sparse reported claims.\n'
    '\n'
    'Recommendation: Expected Claims for AY 2025. The caveat is that if the\n'
    'ELR is incorrect, the error affects every AY uniformly with no\n'
    'correcting signal from actual claims data.\n'
    '\n'
    'KEY TRAP: Do not say Expected Claims is always better for immature years\n'
    'without acknowledging the ELR selection risk.'
)


## Practice Problem 3: Expected Claims as the Foundation for BF

**Exam-style question (Format 3 — BF connection, 5–6 points):**

An actuary uses the following data for a workers compensation book:

| AY   | Earned Premium (000s) | Reported to Date (000s) | % Paid (from CDF) |
|------|----------------------:|------------------------:|-------------------|
| 2023 | 30,000                | 9,600                   | 40.0%             |
| 2024 | 33,000                | 6,600                   | 25.0%             |
| 2025 | 36,000                | 3,600                   | 12.5%             |

Selected ELR: **0.700**

**(a)** Calculate the Expected Claims ultimate for each AY.  
**(b)** Calculate % Unreported for each AY.  
**(c)** Using the BF formula, calculate the BF ultimate for each AY.  
**(d)** Show numerically that if % Paid = 0%, BF converges to Expected Claims. If % Paid = 100%, BF converges to Chain Ladder.  
**(e)** In 2–3 sentences: explain why BF is usually preferred over pure Expected Claims for AY 2023 (40% paid). *(2 pts)*


In [ ]:
import pandas as pd

# PRACTICE PROBLEM 3 SOLUTION
elr = 0.700

data = {
    'AY':         [2023,  2024,  2025],
    'Premium':    [30000, 33000, 36000],
    'Reported':   [9600,  6600,  3600],
    'Pct_Paid':   [0.400, 0.250, 0.125],
}
df = pd.DataFrame(data).set_index('AY')

# PART (a): Expected Claims ultimate
df['EC_Ultimate'] = df['Premium'] * elr

# PART (b): % Unreported = 1 - % Paid
df['Pct_Unreported'] = 1 - df['Pct_Paid']

# PART (c): BF Ultimate = Reported + (% Unreported x EC Ultimate)
df['BF_Ultimate'] = df['Reported'] + df['Pct_Unreported'] * df['EC_Ultimate']

# IBNR from each method
df['EC_IBNR'] = df['EC_Ultimate'] - df['Reported']
df['BF_IBNR'] = df['BF_Ultimate'] - df['Reported']

print('=== PARTS (a), (b), (c): EC AND BF PROJECTION ===')
print(f'  Selected ELR: {elr:.3f}')
print()
cols = ['Premium', 'Reported', 'Pct_Paid', 'Pct_Unreported',
        'EC_Ultimate', 'BF_Ultimate', 'EC_IBNR', 'BF_IBNR']
print(df[cols].to_string(float_format=lambda x: f'{x:,.3f}' if abs(x) < 2 else f'{x:,.0f}'))
print()

print('=== PART (d): BF LIMITING CASES ===')
prem_example = 30000
rep_example  = 9600
ec_ult       = prem_example * elr

# CL ultimate (needs CDF = 1/pct_paid; for 40% paid, CDF = 2.5)
cdf_example = 1 / 0.400
cl_ult = rep_example * cdf_example

# BF at 0% paid (new AY)
bf_0pct = 0 + (1 - 0) * ec_ult          # = EC Ultimate
# BF at 100% paid (fully developed)
bf_100pct = rep_example + (1 - 1) * ec_ult   # = Reported = CL Ultimate when fully dev

print(f'  Example AY: Premium={prem_example:,}  ELR={elr}  EC Ultimate={ec_ult:,}')
print(f'  Reported={rep_example:,}  CDF={cdf_example:.3f}  CL Ultimate={cl_ult:,}')
print()
print(f'  At % Paid = 0%:   BF = Reported + 1.0 x EC_Ult = 0 + {ec_ult:,} = {bf_0pct:,}')
print(f'    -> BF = Expected Claims Ultimate (100% weight on prior)')
print()
print(f'  At % Paid = 100%: BF = Reported + 0.0 x EC_Ult = {rep_example:,} + 0 = {bf_100pct:,}')
print(f'    -> BF = Reported = CL Ultimate (when fully developed, CDF=1.000)')
print()

print('=== PART (e): WHY BF > EXPECTED CLAIMS FOR 40% PAID AY ===')
print(
    'MODEL ANSWER:\n'
    'At 40% paid, AY 2023 has enough actual claims emergence to provide partial\n'
    'credibility -- ignoring it entirely (as Expected Claims does) wastes\n'
    'available information.\n'
    '\n'
    'BF gives 40% weight to actual emergence and 60% weight to the a priori\n'
    'ELR, blending both signals. This produces a more stable estimate than\n'
    'pure Chain Ladder (too leveraged at 40% paid) while still responding\n'
    'to actual experience (unlike pure Expected Claims).\n'
    '\n'
    'Expected Claims is the right choice only when % paid is very small\n'
    '(~0%) or there is no credible historical development pattern at all.'
)


---
## When Expected Claims Is Appropriate — Complete Reference

Pure conceptual question type. Know the situations, the rationale, and the direction of risk if the method is misapplied.

### Expected Claims Assumes:
1. The selected ELR **reliably represents the long-run loss ratio** for each AY
2. Earned premium is on a **consistent rate and coverage basis** (or level-adjusted if not)
3. **Actual claims emergence has little predictive value** for ultimate — true only for very immature or very volatile data
4. **Equal weight** to the a priori assumption regardless of development age

### Situations Where Expected Claims Is Preferred:

| Situation | Why EC Is Appropriate | Risk If ELR Is Wrong |
|---|---|:---:|
| **Very immature AY (≤24 months)** | Chain Ladder CDF is huge; any LDF error is amplified; a priori more reliable than sparse data | ELR error flows directly and linearly to IBNR |
| **New line or territory** | No development history exists; EC is the only credible method | Entire reserve depends on ELR quality |
| **Major operational change** | Historical development patterns don’t transfer; prior disrupted | ELR may also be stale if based on old operations |
| **Rapid premium growth** | Recent AYs are thin; CL leverage is severe for large books with immature data | Flat ELR misses any loss ratio trend in growth |
| **Highly volatile development** | Shock losses or settlement changes distort LDFs; EC unaffected | EC provides no correcting signal if loss ratio shifted |
| **Sparse triangle data** | Too few data points for credible LDF selection | ELR has no anchor if line is truly new |

### When to Avoid Expected Claims:
- AY is **well-developed (≥60 months)** — Chain Ladder is more credible; EC ignores all emergence
- Stable, **credible development history** with no operational changes — CL better
- **ELR is stale** or inconsistent with current pricing — EC error is uniform across all AYs
- Rapidly changing loss ratios — flat ELR systematically misstates immature AYs

### Exam-Ready Answer Template:
> “The Expected Claims method is appropriate here because [specific situation] means the actual claims data is **not credible for predicting ultimate**. The a priori ELR of [X%] is a more reliable anchor than the observed [Y] in reported losses. The key risk is that if the ELR is [too high / too low], the method will **[overstate / understate]** IBNR for every accident year uniformly with no correcting signal from actual emergence.”

### The Credibility Spectrum — Exam-Required Understanding:

```
Less Mature <---- Development Age ----> More Mature

Expected Claims     Bornhuetter-Ferguson     Chain Ladder
(100% prior)           (blend)              (100% emergence)

% Paid ~ 0%          % Paid ~ 50%           % Paid ~ 100%
BF = EC Ultimate    BF blends both         BF = Reported (= CL)
```


---
## Exam-Style Written Answer Examples — Judgment Questions

### Scoring Reminder
Examiners score on **content points**, not length. A 3-sentence answer that addresses mechanism + direction + implication outscores a paragraph that only restates numbers.

> **Common examiner penalty:** *“Candidates stated the ELR without explaining how it was selected or justified. No credit for justification portion.”*

---

### Example A — Compare Methods for an Immature AY

**Question:** “For AY 2025 (12 months of development), the Chain Ladder produces an ultimate of $16,128 and the Expected Claims method produces $14,280. Compare the two methods and explain which you would recommend.” *(3 points)*

**Weak answer (1/3 points):**
> “I would use Expected Claims because the Chain Ladder is higher.”

**Strong answer (3/3 points):**
> “Chain Ladder applies a large CDF (approximately 3.2×) to sparse 12-month reported claims, meaning any LDF selection error is heavily amplified. At this maturity, the method is highly leveraged and unreliable.
>
> Expected Claims ignores immature data and relies entirely on the a priori ELR, which is more credible than 12 months of reported claims. I would recommend Expected Claims for AY 2025, with the caveat that if the ELR is incorrect, the error propagates directly to IBNR with no correcting mechanism from actual data.”

**Why it works:** Names the mechanism (leverage), states the direction of risk, and acknowledges the ELR caveat.

---

### Example B — Recommend a Method for a New Line

**Question:** “A company is writing a new product with no prior development history. Recommend a reserving method and briefly justify.” *(2 points)*

**Strong answer:**
> “I would recommend the Expected Claims method. Because there is no historical development pattern, the Chain Ladder cannot be applied — LDFs require at least one diagonal of prior data.
>
> The Expected Claims method uses a selected ELR (from pricing assumptions, industry benchmarks, or similar lines) to project ultimates, avoiding the leverage problem entirely. The primary risk is that if the selected ELR does not reflect actual loss experience, the IBNR will be misstated for every accident year uniformly.”

---

### Example C — Impact of ELR Change

**Question:** “The actuary increases the selected ELR by 5 percentage points (e.g., from 0.70 to 0.75). Describe the impact on reserves.” *(2 points)*

**Weak answer (0/2 points):**
> “Reserves will increase.”

**Strong answer (2/2 points):**
> “Under the Expected Claims method, IBNR has a direct, linear relationship with the ELR: IBNR = (ELR − Reported/Premium) × Premium. A 5-point increase in ELR increases expected ultimate for **every** accident year by exactly 5% of earned premium, regardless of maturity.
>
> This contrasts with Chain Ladder, where LDF errors propagate differently by maturity (immature AYs amplify errors more). With Expected Claims, the reserve impact is uniform and immediate — there is no ‘self-correcting’ mechanism.”

---

### Example D — Why Prefer BF Over Expected Claims?

**Question:** “An accident year is 40% paid. Why might you prefer the Bornhuetter-Ferguson method over the Expected Claims method?” *(2 points)*

**Strong answer:**
> “At 40% paid, the accident year has enough actual claims emergence to provide partial credibility — ignoring it entirely (as Expected Claims does) wastes available information.
>
> BF blends 40% weight on actual emergence with 60% weight on the a priori ELR, incorporating both signals. This produces a more balanced estimate: more stable than pure Chain Ladder at this maturity, but more responsive to actual experience than pure Expected Claims. Expected Claims is only preferable when % paid is very low (near zero) or when actual claims data is not credible.”

---

### Key Phrases Examiners Reward:
- *“…a priori ELR is more credible than immature reported claims…”*
- *“…Chain Ladder is highly leveraged at early maturities…”*
- *“…linear, uniform relationship between ELR and IBNR…”*
- *“…no self-correcting mechanism from actual claims emergence…”*
- *“…if ELR is [too high/low], IBNR will be [overstated/understated] for every AY…”*
- *“…BF is preferred at moderate maturities because it blends both signals…”*


---
## Exam Trap Awareness — Expected Claims Method

From CAS Examiner Reports and general exam patterns.

### Mechanical Traps

| Trap | What Goes Wrong | How to Avoid |
|---|---|---|
| **IBNR = Ultimate, not difference** | Reporting Expected Ultimate as IBNR | Always: IBNR = Expected Ultimate − Reported (or − Paid) |
| **Wrong loss basis** | Using paid when question specifies reported basis (or vice versa) | Read the question: reported basis IBNR is smaller than paid basis |
| **ELR applied to wrong base** | Multiplying ELR by reported losses instead of earned premium | Ultimate = ELR × **Earned Premium**, not × Reported |
| **Forgetting level adjustment** | Using raw historical ECR without adjusting for rate changes | If ECR derived from historical data at different rate levels, apply a level adjustment factor |
| **ECR from immature AYs** | Deriving ECR using AYs that are not yet fully developed | Use only mature AYs (≥84 months for auto liability) to calibrate ECR |
| **Flat ELR ignores trend** | Applying same ELR to all AYs when loss ratios are trending | Consider year-specific ELRs or note this as a limitation of the flat assumption |

### Judgment / Written Answer Traps

| Trap | What Goes Wrong | How to Avoid |
|---|---|---|
| **No ELR risk caveat** | “Expected Claims is always better for immature years” | Always pair the recommendation with: ‘the risk is that if ELR is wrong…’ |
| **No direction of impact** | Saying ELR is uncertain without stating which way | Always say: a higher ELR overstates / lower ELR understates IBNR |
| **Confusing EC with BF** | Treating them as identical | EC = 100% prior; BF = blend. They produce the same answer only at 0% paid |
| **Not justifying ELR selection** | Stating ELR without explaining how it was chosen | Always name the derivation: volume-weighted historical average, pricing assumption, etc. |
| **Ignoring exposure growth** | Applying flat ELR while premiums are growing rapidly | Rapid growth may distort the ELR; note this as a potential bias |
| **Overly generic answers** | “Expected Claims is stable” with no further detail | Name what it’s stable against: LDF errors, case reserve changes, diagonal effects |

---

### Quick Self-Check Before Finalizing Any Expected Claims Answer:

- [ ] Did I compute **Ultimate = ELR × Earned Premium** (not × Reported)?
- [ ] Did I compute **IBNR = Expected Ultimate − Reported** (not just state Ultimate)?
- [ ] Did I use **reported basis or paid basis** as the question specifies?
- [ ] Did I **justify the ELR selection** (volume-weighted, pricing assumption, etc.)?
- [ ] Did I apply a **level adjustment** if historical ECR came from a different rate period?
- [ ] If recommending EC over CL — did I **acknowledge the ELR selection risk**?
- [ ] If discussing ELR uncertainty — did I state the **direction of impact**?
- [ ] For ‘briefly’ questions — did I keep my answer to **2–3 sentences**?


---
## Method Comparison Quick Reference

Use this table when an exam question asks you to compare methods or choose between them.

| Dimension | Chain Ladder | Expected Claims | Bornhuetter-Ferguson |
|---|:---:|:---:|:---:|
| **Data dependency** | 100% emergence | 0% emergence | Blend (q vs 1−q) |
| **Prior (ELR) dependency** | 0% | 100% | Blend (1−q weight) |
| **Stability for immature AYs** | Low | High | Moderate |
| **Sensitivity to bad LDFs** | High (amplified by CDF) | None | Partial |
| **Sensitivity to bad ELR** | None | High (linear, uniform) | Partial |
| **Self-correcting as AY matures** | Yes — LDFs stabilize | No | Yes — converges to CL |
| **Best for** | Mature, stable AYs | Very immature / new lines | Middle maturity |
| **Exam default for ≤24 months** | No | Yes | Maybe |
| **Exam default for 24–60 months** | No | Rarely | Yes |
| **Exam default for ≥60 months** | Yes | No | Optional |

### Convergence Rules (must know for BF questions):
- **% Paid → 0%:** BF Ultimate → Expected Claims Ultimate (= ELR × Premium)
- **% Paid → 100%:** BF Ultimate → Reported = Chain Ladder Ultimate (when CDF = 1.000)
- **At any % paid:** BF = Reported + (% Unreported × Expected Ultimate)

### Classic 7-Point Exam Question Structure:
```
(a) Calculate CL ultimate for all AYs             [3 pts] -- mechanical
(b) Calculate Expected Claims ultimate for all AYs [2 pts] -- mechanical
(c) Compare results; recommend method for newest AY [2 pts] -- judgment
```
Part (c) is where most points are lost. Strong answers always:
1. Name the maturity of the newest AY (% paid / development age)
2. Name the leverage risk of Chain Ladder at that maturity
3. Name the ELR-selection risk of Expected Claims
4. State a clear recommendation with a brief caveat
